In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Cross-Catalog Comparison: Italy (INGV) vs US (USGS) vs Japan (JMA)

Extends cross_catalog_comparison.py to a three-catalog analysis.
Run the individual INGV, USGS and JAPAN notebooks first so the pickle
caches and metrics CSVs exist; set USE_CACHED_NETWORKS = False to
force a full rebuild from raw CSVs.

Run as a script  : python comparison_it_us_japan.py
Convert to notebook: python convert_to_notebook.py comparison_it_us_japan.py notebooks/comparison_it_us_japan.ipynb
"""

import logging
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
import seaborn as sns
from scipy.stats import linregress

from src.assortativity import attach_catalog_attrs, compute_assortativity
from src.link_prediction import split_edges_temporal, evaluate_predictors, plot_auc_comparison
from src.metrics import estimate_gamma_mle
from src.network import build_abe_suzuki_network
from src.robustness import simulate_robustness
from src.community import run_louvain, run_infomap, compute_nmi_matrix
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings("ignore")

DATA_DIR_IT = Path("data/INGV")
DATA_DIR_US = Path("data/USGS")
DATA_DIR_JP = Path("data/Japan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)

TARGET_CRS_IT = "epsg:32632"   # UTM 32N — Italy
TARGET_CRS_US = "epsg:5070"    # CONUS Albers Equal Area — US
TARGET_CRS_JP = "epsg:32654"   # UTM 54N — Japan

K_MIN_FIT    = 10
N_ROB_CHKPTS = 30
SEED         = 42
LP_SAMPLE    = 5_000
SAVE_PDF: bool = True
SAVE_JPG: bool = True

# Colour palette — consistent across all figures
C_IT = "steelblue"
C_US = "darkorange"
C_JP = "seagreen"

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "comparison")

USE_CACHED_NETWORKS: bool = True

_METRICS = [
    "Degree", "PageRank", "Closeness", "Betweenness",
    "Eigenvector", "Katz", "HITS_Hub", "HITS_Auth",
]

## Data Loading — Italy (INGV)

In [ ]:
print("Loading Italy catalog...")
df_it = pd.read_csv(DATA_DIR_IT / "italy_earthquakes_1985_2025.csv")
df_it["time"] = pd.to_datetime(df_it["time"], utc=True)
df_it = df_it.rename(columns={"depth": "depth_km"}) if "depth" in df_it.columns else df_it
df_it["year"] = df_it["time"].dt.year
print(f"  {len(df_it):,} events  M [{df_it['magnitude'].min():.1f}, {df_it['magnitude'].max():.1f}]")

## Data Loading — US (USGS)

In [ ]:
print("Loading US catalog...")
df_us = pd.read_csv(DATA_DIR_US / "us_earthquakes_1985_2025.csv")
df_us["time"] = pd.to_datetime(df_us["time"], utc=True)
df_us = df_us.rename(columns={"depth": "depth_km"}) if "depth" in df_us.columns else df_us
df_us["year"] = df_us["time"].dt.year
print(f"  {len(df_us):,} events  M [{df_us['magnitude'].min():.1f}, {df_us['magnitude'].max():.1f}]")

## Data Loading — Japan (JMA)

In [ ]:
print("Loading Japan catalog (JMA via ISC)...")
df_jp = pd.read_csv(DATA_DIR_JP / "japan_earthquakes_jma_1985_2025_m1_5.csv")
df_jp["time"] = pd.to_datetime(df_jp["time"], utc=True)
df_jp["year"] = df_jp["time"].dt.year
print(f"  {len(df_jp):,} events  M [{df_jp['magnitude'].min():.1f}, {df_jp['magnitude'].max():.1f}]")

## Network Construction

All three 10 km Abe-Suzuki networks are loaded from pickle cache (if available)
or built from the raw CSVs. The cache is generated by the individual catalog
notebooks (ITALY_network_ABE, US_network_ABE, JAPAN_network_ABE).

Japan is the **original catalog** from Abe & Suzuki (2004), making this
comparison both a cross-regional study and a direct replication check:
Italy and US extend the methodology; Japan validates it against the paper.

In [ ]:
_CACHE = {
    "G_it_5":  RESULTS_DIR / "cache" / "italy_G5km.pkl",
    "G_it_10": RESULTS_DIR / "cache" / "italy_G10km.pkl",
    "G_us_5":  RESULTS_DIR / "cache" / "us_G5km.pkl",
    "G_us_10": RESULTS_DIR / "cache" / "us_G10km.pkl",
    "G_jp_5":  RESULTS_DIR / "cache" / "japan_G5km.pkl",
    "G_jp_10": RESULTS_DIR / "cache" / "japan_G10km.pkl",
}

def _load_or_build(key: str, df: pd.DataFrame, cell_size_km: float,
                   target_crs: str) -> nx.DiGraph:
    path = _CACHE[key]
    if USE_CACHED_NETWORKS and path.exists():
        print(f"  Loading {key} from cache ({path.name})...")
        with open(path, "rb") as f:
            return pickle.load(f)
    verb = "Rebuilding" if path.exists() else "Building"
    print(f"  {verb} {key} from raw catalog...")
    return build_abe_suzuki_network(
        df.sort_values("time").reset_index(drop=True),
        cell_size_km=cell_size_km, target_crs=target_crs,
    )

print(f"\nNetwork construction (USE_CACHED_NETWORKS={USE_CACHED_NETWORKS})...")
G_it_5  = _load_or_build("G_it_5",  df_it, 5,  TARGET_CRS_IT)
G_it_10 = _load_or_build("G_it_10", df_it, 10, TARGET_CRS_IT)
G_us_5  = _load_or_build("G_us_5",  df_us, 5,  TARGET_CRS_US)
G_us_10 = _load_or_build("G_us_10", df_us, 10, TARGET_CRS_US)
G_jp_5  = _load_or_build("G_jp_5",  df_jp, 5,  TARGET_CRS_JP)
G_jp_10 = _load_or_build("G_jp_10", df_jp, 10, TARGET_CRS_JP)

def _giant_undirected(G: nx.DiGraph) -> nx.Graph:
    G_und = G.to_undirected()
    G_und.remove_edges_from(nx.selfloop_edges(G_und))
    gcc_nodes = max(nx.connected_components(G_und), key=len)
    return G_und.subgraph(gcc_nodes).copy()

G_it_10_gcc = _giant_undirected(G_it_10)
G_us_10_gcc = _giant_undirected(G_us_10)
G_jp_10_gcc = _giant_undirected(G_jp_10)

for label, G in [("Italy", G_it_10_gcc), ("US", G_us_10_gcc), ("Japan", G_jp_10_gcc)]:
    print(f"  {label} 10 km GCC: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## Catalog Summary Table

Three-catalog overview: event counts, magnitude and depth ranges,
and 10 km network statistics. Japan has the highest seismicity rate
among the three (subduction + intraplate + volcanic), the widest
depth range (crustal to > 600 km deep-slab), and was the original
catalog used by Abe & Suzuki (2004) — making it the replication benchmark.

In [ ]:
def _gcc_fraction(G: nx.DiGraph) -> float:
    G_und = G.to_undirected()
    G_und.remove_edges_from(nx.selfloop_edges(G_und))
    if G_und.number_of_nodes() == 0:
        return 0.0
    gcc = max(nx.connected_components(G_und), key=len)
    return len(gcc) / G_und.number_of_nodes()

summary = pd.DataFrame(
    {
        "Italy (INGV)": [
            f"{len(df_it):,}",
            f"{df_it['time'].min().year}–{df_it['time'].max().year}",
            f"{df_it['magnitude'].min():.1f}–{df_it['magnitude'].max():.1f}",
            f"{df_it['depth_km'].min():.0f}–{df_it['depth_km'].max():.0f}",
            f"{G_it_10.number_of_nodes():,}",
            f"{G_it_10.number_of_edges():,}",
            f"{_gcc_fraction(G_it_10):.1%}",
        ],
        "US (USGS)": [
            f"{len(df_us):,}",
            f"{df_us['time'].min().year}–{df_us['time'].max().year}",
            f"{df_us['magnitude'].min():.1f}–{df_us['magnitude'].max():.1f}",
            f"{df_us['depth_km'].min():.0f}–{df_us['depth_km'].max():.0f}",
            f"{G_us_10.number_of_nodes():,}",
            f"{G_us_10.number_of_edges():,}",
            f"{_gcc_fraction(G_us_10):.1%}",
        ],
        "Japan (JMA)": [
            f"{len(df_jp):,}",
            f"{df_jp['time'].min().year}–{df_jp['time'].max().year}",
            f"{df_jp['magnitude'].min():.1f}–{df_jp['magnitude'].max():.1f}",
            f"{df_jp['depth_km'].min():.0f}–{df_jp['depth_km'].max():.0f}",
            f"{G_jp_10.number_of_nodes():,}",
            f"{G_jp_10.number_of_edges():,}",
            f"{_gcc_fraction(G_jp_10):.1%}",
        ],
    },
    index=[
        "Events", "Period", "Magnitude range", "Depth range (km)",
        "Nodes (10 km)", "Edges (10 km)", "GCC fraction (10 km)",
    ],
)
print("\n=== Catalog Summary ===")
display(summary)

## Degree Distribution Overlay

Visual comparison of log-binned $P(k)$ tails for all three catalogs on the
same log-log axes. If Italy, US, and Japan collapse onto a single straight
line, the power-law topology is **universal** — a property of fault-network
geometry transcending tectonic setting, seismicity rate, and monitoring agency.

Japan is the reference: Abe & Suzuki (2004) reported $\gamma \approx 2.2$.
Agreement between the three fitted slopes would confirm universality;
divergence would suggest tectonic-regime dependence.

In [ ]:
def _log_binned(G: nx.Graph, n_bins: int = 25) -> tuple[np.ndarray, np.ndarray]:
    degs = np.array([d for _, d in G.degree(weight="weight") if d > 0], dtype=float)
    bins = np.logspace(np.log10(degs.min()), np.log10(degs.max()), n_bins)
    counts, edges = np.histogram(degs, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    widths   = np.diff(edges)
    Pk = counts / (len(degs) * widths)
    valid = Pk > 0
    return centers[valid], Pk[valid]

k_it, P_it = _log_binned(G_it_10_gcc)
k_us, P_us = _log_binned(G_us_10_gcc)
k_jp, P_jp = _log_binned(G_jp_10_gcc)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(k_it, P_it, color=C_IT, alpha=0.8, edgecolors="k", s=55, label="Italy (INGV)")
ax.scatter(k_us, P_us, color=C_US, alpha=0.8, edgecolors="k", s=55, label="US (USGS)")
ax.scatter(k_jp, P_jp, color=C_JP, alpha=0.8, edgecolors="k", s=55, label="Japan (JMA)")

for k_vals, P_vals, col in [
    (k_it, P_it, C_IT), (k_us, P_us, C_US), (k_jp, P_jp, C_JP)
]:
    mask = k_vals >= K_MIN_FIT
    if mask.sum() > 2:
        slope, intercept, *_ = linregress(np.log10(k_vals[mask]), np.log10(P_vals[mask]))
        fit = 10**intercept * k_vals[mask] ** slope
        ax.plot(k_vals[mask], fit, "--", color=col, linewidth=2,
                label=rf"Fit $\gamma={-slope:.2f}$")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Degree $k$", fontsize=13)
ax.set_ylabel("Probability Density $P(k)$", fontsize=13)
ax.set_title("Degree Distribution Overlay — Italy / US / Japan (10 km, GCC)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
savefig("3cat_degree_distribution_overlay")
plt.show()

## Power-Law Exponent Comparison

MLE $\gamma$ at both 5 km and 10 km cell sizes for all three catalogs.
Abe & Suzuki (2004) found $\gamma \approx 2.2$ for the Japan JMA catalog —
this is the primary replication target. $\gamma < 3$ implies a divergent
second moment $\langle k^2 \rangle$; $\gamma < 2$ implies a divergent mean.

Cell-size sensitivity: if $\gamma$ shifts significantly from 5 km to 10 km,
the power-law is resolution-dependent and may not reflect an intrinsic
property of the fault system.

In [ ]:
def _gamma(G: nx.Graph) -> float:
    degs = [d for _, d in G.degree(weight="weight") if d >= K_MIN_FIT]
    return estimate_gamma_mle(degs, k_min=K_MIN_FIT) if len(degs) > 5 else float("nan")

gamma_vals = {
    "Italy 5 km":  _gamma(_giant_undirected(G_it_5)),
    "Italy 10 km": _gamma(G_it_10_gcc),
    "US 5 km":     _gamma(_giant_undirected(G_us_5)),
    "US 10 km":    _gamma(G_us_10_gcc),
    "Japan 5 km":  _gamma(_giant_undirected(G_jp_5)),
    "Japan 10 km": _gamma(G_jp_10_gcc),
}
df_gamma = pd.DataFrame(list(gamma_vals.items()), columns=["Network", "γ (MLE)"])
print("\n=== Power-Law Exponent γ ===")
display(df_gamma.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
colors = [C_IT, C_IT, C_US, C_US, C_JP, C_JP]
alphas = [1.0, 0.55, 1.0, 0.55, 1.0, 0.55]
bars = ax.bar(df_gamma["Network"], df_gamma["γ (MLE)"],
              color=colors, alpha=0.85, edgecolor="k", linewidth=0.8)
for bar, val in zip(bars, df_gamma["γ (MLE)"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax.axhline(2.0, color="gray",   ls="--", lw=1.2, label="γ = 2 (BA scale-free)")
ax.axhline(2.2, color="seagreen", ls=":",  lw=1.5, alpha=0.7,
           label="γ ≈ 2.2 (Abe & Suzuki 2004 reference)")
ax.axhline(3.0, color="silver", ls=":",  lw=1.2, label="γ = 3 (classical bound)")
ax.set_ylabel("γ (MLE)", fontsize=12)
ax.set_title("Power-Law Exponent Comparison: Italy / US / Japan", fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, max(df_gamma["γ (MLE)"].max() * 1.2, 3.5))
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
savefig("3cat_gamma_comparison_bar")
plt.show()

## Macroscopic Metrics Comparison

Small-world test for all three catalogs simultaneously (Watts & Strogatz 1998).
Path length is estimated by sampling 300 source nodes.

Japan's extremely high seismicity rate should produce a denser network
(higher $\langle k \rangle$) and potentially shorter average path length
than Italy or the US. The 2011 Tohoku sequence alone generated tens of
thousands of cells, likely inflating the giant component fraction.

In [ ]:
def _path_length_sampled(G: nx.Graph, k: int = 300, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    nodes = list(G.nodes())
    sample = rng.choice(nodes, size=min(k, len(nodes)), replace=False).tolist()
    lengths = []
    for s in sample:
        sp = nx.single_source_shortest_path_length(G, s)
        lengths.extend(sp.values())
    return float(np.mean(lengths)) if lengths else float("nan")

print("\nComputing macroscopic metrics (~1 min)...")
rows = []
for label, G_full, G_gcc in [
    ("Italy 10 km",  G_it_10, G_it_10_gcc),
    ("US 10 km",     G_us_10, G_us_10_gcc),
    ("Japan 10 km",  G_jp_10, G_jp_10_gcc),
]:
    n = G_gcc.number_of_nodes()
    m = G_gcc.number_of_edges()
    rows.append({
        "Network":        label,
        "N (GCC)":        n,
        "M (GCC)":        m,
        "Density":        round(nx.density(G_gcc), 5),
        "⟨k⟩":           round(2 * m / n, 2),
        "C (clustering)": round(nx.average_clustering(G_gcc), 4),
        "L (sampled)":    round(_path_length_sampled(G_gcc, k=300, seed=SEED), 3),
        "GCC fraction":   round(_gcc_fraction(G_full), 4),
    })

df_macro = pd.DataFrame(rows).set_index("Network")
print("\n=== Macroscopic Metrics ===")
display(df_macro)

## Assortativity Comparison

Side-by-side Newman (2003) assortativity $r$ for degree, depth, and magnitude
across all three catalogs. **Disassortative degree** ($r < 0$) is the canonical
scale-free signature and is expected in all three.

**Depth assortativity** is most informative for Japan, which has three
seismically distinct depth regimes (shallow crustal, slab interface, deep slab).
Strong positive depth assortativity would imply tectonic layers generate
topologically self-contained sequences; near-zero implies cross-depth triggering.

In [ ]:
print("\nAttaching catalog attributes to nodes...")
attach_catalog_attrs(G_it_10, df_it, cell_size_km=10.0, target_crs=TARGET_CRS_IT)
attach_catalog_attrs(G_us_10, df_us, cell_size_km=10.0, target_crs=TARGET_CRS_US)
attach_catalog_attrs(G_jp_10, df_jp, cell_size_km=10.0, target_crs=TARGET_CRS_JP)

df_assort_it = compute_assortativity(G_it_10)[["r"]].rename(columns={"r": "Italy"})
df_assort_us = compute_assortativity(G_us_10)[["r"]].rename(columns={"r": "US"})
df_assort_jp = compute_assortativity(G_jp_10)[["r"]].rename(columns={"r": "Japan"})
df_assort = df_assort_it.join(df_assort_us).join(df_assort_jp)
print("\n=== Assortativity Coefficients ===")
display(df_assort.round(4))

attrs = df_assort.index.tolist()
x = np.arange(len(attrs))
w = 0.25

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w, df_assort["Italy"],  w, color=C_IT, alpha=0.85, edgecolor="k", label="Italy (INGV)")
ax.bar(x,     df_assort["US"],     w, color=C_US, alpha=0.85, edgecolor="k", label="US (USGS)")
ax.bar(x + w, df_assort["Japan"],  w, color=C_JP, alpha=0.85, edgecolor="k", label="Japan (JMA)")
ax.axhline(0, color="k", lw=0.8)
ax.axhline(-0.05, color="gray", lw=0.8, ls="--", alpha=0.5)
ax.axhline( 0.05, color="gray", lw=0.8, ls="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(attrs, fontsize=11)
ax.set_ylabel("Assortativity coefficient r", fontsize=12)
ax.set_title("Assortativity Comparison: Italy / US / Japan (10 km)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis="y", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_assortativity_comparison_bar")
plt.show()

## Centrality Distribution Comparison

Violin plots of centrality score distributions per metric across all three
catalogs on a log scale. The heavy right tail confirms scale-free centrality
concentration in all three tectonic settings.

**Hub depth comparison** across three catalogs tests the tectonic-regime
hypothesis: Italy hubs should be shallow (Apennine crustal, 5–15 km);
US hubs may be near-surface (induced seismicity, Geysers);
Japan hubs may be intermediate depth (slab interface, 20–50 km).

In [ ]:
from src.centrality import compute_all_centralities

path_it = RESULTS_DIR / "data" / "italy_eq_network_metrics.csv"
path_us = RESULTS_DIR / "data" / "us_eq_network_metrics.csv"
path_jp = RESULTS_DIR / "data" / "japan_eq_network_metrics.csv"

def _load_or_compute(path: Path, G: nx.DiGraph, label: str) -> pd.DataFrame:
    if path.exists():
        print(f"  Loading {path.name}")
        return pd.read_csv(path)
    print(f"  {path.name} not found — computing centrality for {label}...")
    return compute_all_centralities(G, k_betweenness=200, seed=SEED)

df_cent_it = _load_or_compute(path_it, G_it_10, "Italy")
df_cent_us = _load_or_compute(path_us, G_us_10, "US")
df_cent_jp = _load_or_compute(path_jp, G_jp_10, "Japan")

available = [m for m in _METRICS
             if m in df_cent_it.columns and m in df_cent_us.columns and m in df_cent_jp.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, metric in enumerate(available):
    ax = axes[i]
    combined = pd.DataFrame({
        "score": pd.concat([df_cent_it[metric], df_cent_us[metric], df_cent_jp[metric]],
                           ignore_index=True),
        "catalog": (["Italy"] * len(df_cent_it) + ["US"] * len(df_cent_us)
                    + ["Japan"] * len(df_cent_jp)),
    })
    combined["score"] = combined["score"].clip(lower=1e-10)
    sns.violinplot(
        data=combined, x="catalog", y="score",
        palette={"Italy": C_IT, "US": C_US, "Japan": C_JP},
        inner="quartile", cut=0, ax=ax, order=["Italy", "US", "Japan"],
    )
    ax.set_yscale("log")
    ax.set_title(metric, fontsize=10, pad=4)
    ax.set_xlabel("")
    ax.set_ylabel("Score", fontsize=8)
    ax.grid(axis="y", ls="--", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Centrality Score Distributions: Italy / US / Japan (10 km, log scale)",
             fontsize=14, y=1.01)
plt.tight_layout()
savefig("3cat_centrality_violin_comparison")
plt.show()

# ── Hub depth comparison ──────────────────────────────────────────────────────
depth_rows = []
for metric in available:
    d_it = df_cent_it.nlargest(10, metric)["depth_km"].mean()
    d_us = df_cent_us.nlargest(10, metric)["depth_km"].mean()
    d_jp = df_cent_jp.nlargest(10, metric)["depth_km"].mean()
    depth_rows.append({"Metric": metric,
                       "Italy top-10 depth (km)": round(d_it, 1),
                       "US top-10 depth (km)":    round(d_us, 1),
                       "Japan top-10 depth (km)": round(d_jp, 1)})
df_depth = pd.DataFrame(depth_rows)
print("\n=== Mean depth of top-10 nodes per metric ===")
display(df_depth)

x  = np.arange(len(available))
w  = 0.25
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - w, df_depth["Italy top-10 depth (km)"], w, color=C_IT, alpha=0.85, edgecolor="k", label="Italy")
ax.bar(x,     df_depth["US top-10 depth (km)"],    w, color=C_US, alpha=0.85, edgecolor="k", label="US")
ax.bar(x + w, df_depth["Japan top-10 depth (km)"], w, color=C_JP, alpha=0.85, edgecolor="k", label="Japan")
ax.set_xticks(x)
ax.set_xticklabels(available, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Mean depth of top-10 nodes (km)", fontsize=11)
ax.set_title("Hub Depth by Centrality Metric: Italy / US / Japan", fontsize=13)
ax.axhline(0, color="black", lw=0.8)
ax.legend(fontsize=10)
ax.grid(axis="y", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_hub_depth_comparison")
plt.show()

## Robustness Comparison

Scale-free fragility curves for all three catalogs: random vs targeted node
removal. Albert et al. (2000): steeper targeted curve → more hub-dominated
topology; flatter random curve → robustness to random failure.

The critical fraction $f_c$ (GCC $< 0.05$ under targeted attack) measures
hub dependence. Japan's massive Tohoku-driven hubs may produce the lowest
$f_c$ of the three — the most fragile topology under targeted monitoring.

In [ ]:
print("\nSimulating robustness (~2 min for three catalogs)...")
rob_it_rand = simulate_robustness(G_it_10_gcc, strategy="random",   n_checkpoints=N_ROB_CHKPTS, seed=SEED)
rob_it_targ = simulate_robustness(G_it_10_gcc, strategy="targeted", n_checkpoints=N_ROB_CHKPTS)
rob_us_rand = simulate_robustness(G_us_10_gcc, strategy="random",   n_checkpoints=N_ROB_CHKPTS, seed=SEED)
rob_us_targ = simulate_robustness(G_us_10_gcc, strategy="targeted", n_checkpoints=N_ROB_CHKPTS)
rob_jp_rand = simulate_robustness(G_jp_10_gcc, strategy="random",   n_checkpoints=N_ROB_CHKPTS, seed=SEED)
rob_jp_targ = simulate_robustness(G_jp_10_gcc, strategy="targeted", n_checkpoints=N_ROB_CHKPTS)

# Three-panel: one per catalog
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, title, r_rand, r_targ in [
    (axes[0], "Italy (INGV)",  rob_it_rand, rob_it_targ),
    (axes[1], "US (USGS)",     rob_us_rand, rob_us_targ),
    (axes[2], "Japan (JMA)",   rob_jp_rand, rob_jp_targ),
]:
    ax.plot(r_rand["fraction_removed"], r_rand["gcc_fraction"],
            "o-", color="steelblue", lw=2, ms=4, label="Random")
    ax.plot(r_targ["fraction_removed"], r_targ["gcc_fraction"],
            "s--", color="crimson",   lw=2, ms=4, label="Targeted")
    ax.set_xlabel("Fraction of nodes removed", fontsize=11)
    ax.set_ylabel("GCC fraction", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(ls="--", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Robustness Comparison: Italy / US / Japan (10 km GCC)", fontsize=14)
plt.tight_layout()
savefig("3cat_robustness_comparison_panels")
plt.show()

# Overlay: targeted attack only
fig, ax = plt.subplots(figsize=(8, 5))
for label, r_targ, r_rand, col in [
    ("Italy", rob_it_targ, rob_it_rand, C_IT),
    ("US",    rob_us_targ, rob_us_rand, C_US),
    ("Japan", rob_jp_targ, rob_jp_rand, C_JP),
]:
    ax.plot(r_targ["fraction_removed"], r_targ["gcc_fraction"],
            "s-", color=col, lw=2, ms=5, label=f"{label} — targeted")
    ax.plot(r_rand["fraction_removed"], r_rand["gcc_fraction"],
            "o--", color=col, lw=1.5, ms=4, alpha=0.5, label=f"{label} — random")
ax.set_xlabel("Fraction of nodes removed", fontsize=12)
ax.set_ylabel("GCC fraction", fontsize=12)
ax.set_title("Robustness Overlay: Italy / US / Japan", fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_robustness_overlay")
plt.show()

## Community Structure Comparison

Louvain and InfoMap community counts, modularity $Q$, and NMI stability
(10 independent Louvain runs) for all three catalogs.

**Expected ordering**: Italy (simplest tectonic setting, fewest communities)
< US (multi-region) < Japan (multi-arc, multi-depth-regime, highest complexity).
InfoMap community counts should scale with catalog size since resolution follows
the flow volume — Japan's high event count implies many small flow communities.

In [ ]:
print("\nRunning community detection on all three GCCs...")
cm_it_lv = run_louvain(G_it_10_gcc, seed=SEED)
cm_us_lv = run_louvain(G_us_10_gcc, seed=SEED)
cm_jp_lv = run_louvain(G_jp_10_gcc, seed=SEED)
cm_it_im = run_infomap(G_it_10_gcc, directed=False, seed=SEED)
cm_us_im = run_infomap(G_us_10_gcc, directed=False, seed=SEED)
cm_jp_im = run_infomap(G_jp_10_gcc, directed=False, seed=SEED)

def _louvain_nmi_stability(G: nx.Graph, n: int = 10, seed: int = 42) -> float:
    runs = {f"run_{i}": {v: c for c, nodes in
                         enumerate(nx.community.louvain_communities(G, seed=seed + i))
                         for v in nodes}
            for i in range(n)}
    nmi = compute_nmi_matrix(runs)
    vals = nmi.values[np.tril_indices_from(nmi.values, k=-1)]
    return float(vals.mean())

def _modularity(G: nx.Graph, cm: dict) -> float:
    communities = [{n for n, c in cm.items() if c == k} for k in set(cm.values())]
    return round(nx.community.modularity(G, communities), 4)

print("  NMI stability Italy...")
nmi_stab_it = _louvain_nmi_stability(G_it_10_gcc, n=10, seed=SEED)
print("  NMI stability US...")
nmi_stab_us = _louvain_nmi_stability(G_us_10_gcc, n=10, seed=SEED)
print("  NMI stability Japan...")
nmi_stab_jp = _louvain_nmi_stability(G_jp_10_gcc, n=10, seed=SEED)

df_comm = pd.DataFrame({
    "Catalog": ["Italy (INGV)", "US (USGS)", "Japan (JMA)"],
    "Louvain k":          [len(set(cm_it_lv.values())), len(set(cm_us_lv.values())), len(set(cm_jp_lv.values()))],
    "InfoMap k":          [len(set(cm_it_im.values())), len(set(cm_us_im.values())), len(set(cm_jp_im.values()))],
    "Louvain Q":          [_modularity(G_it_10_gcc, cm_it_lv),
                           _modularity(G_us_10_gcc, cm_us_lv),
                           _modularity(G_jp_10_gcc, cm_jp_lv)],
    "NMI stability":      [round(nmi_stab_it, 3), round(nmi_stab_us, 3), round(nmi_stab_jp, 3)],
})
print("\n=== Community Structure ===")
display(df_comm)

# Bar chart: community counts (log scale)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.35
catalogs = ["Italy", "US", "Japan"]
colors3  = [C_IT, C_US, C_JP]
bars_lv = ax.bar(x - w/2, df_comm["Louvain k"], w,
                 color=colors3, alpha=0.85, edgecolor="k", label="Louvain")
bars_im = ax.bar(x + w/2, df_comm["InfoMap k"], w,
                 color=colors3, alpha=0.45, edgecolor="k", hatch="//", label="InfoMap")
for bar in list(bars_lv) + list(bars_im):
    h = int(bar.get_height())
    ax.text(bar.get_x() + bar.get_width() / 2, h * 1.15, str(h),
            ha="center", va="bottom", fontsize=9)
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(["Italy (INGV)", "US (USGS)", "Japan (JMA)"], fontsize=11)
ax.set_ylabel("Number of communities (log scale)", fontsize=12)
ax.set_title("Community Count: Louvain vs InfoMap — 3 Catalogs", fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis="y", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_community_count_comparison")
plt.show()

# Geographic community maps — all three catalogs
_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
_US_BOUNDS = dict(west=-125.0, east=-65.0, south=24.6, north=50.0)
_JP_BOUNDS = dict(west=122, east=154, south=24, north=46.5)

for cat_label, G_gcc, cm_lv, bounds, center_lat, center_lon, height, save_name in [
    ("Italy",  G_it_10_gcc, cm_it_lv, _IT_BOUNDS, 41.9, 12.5,  700, "3cat_community_louvain_italy"),
    ("US",     G_us_10_gcc, cm_us_lv, _US_BOUNDS, 37.5, -96.0, 600, "3cat_community_louvain_us"),
    ("Japan",  G_jp_10_gcc, cm_jp_lv, _JP_BOUNDS, 36.0, 138.0, 900, "3cat_community_louvain_japan"),
]:
    df_map = pd.DataFrame([
        {"cell_id": n, "community": str(cm_lv[n]),
         "lat": G_gcc.nodes[n]["lat"], "lon": G_gcc.nodes[n]["lon"],
         "degree": G_gcc.degree(n)}
        for n in G_gcc.nodes() if "lat" in G_gcc.nodes[n]
    ])
    df_map = df_map[df_map["community"].isin(
        df_map["community"].value_counts()[lambda s: s >= 50].index
    )]
    fig_map = px.scatter_map(
        df_map, lat="lat", lon="lon",
        color="community", size="degree", size_max=18,
        color_discrete_sequence=px.colors.qualitative.Bold,
        hover_name="community",
        map_style="carto-positron",
        title=f"Seismic Communities — Louvain ({cat_label})",
    )
    fig_map.update_traces(marker=dict(opacity=0.7))
    fig_map.update_layout(
        margin={"r": 0, "t": 40, "l": 0, "b": 0},
        height=height,
        map=dict(center={"lat": center_lat, "lon": center_lon}, zoom=0, bounds=bounds),
    )
    save_plotly(fig_map, save_name)
    fig_map.show()

## Temporal Evolution of γ(t)

Year-by-year MLE $\hat{\gamma}$ for all three catalogs tests the Abe & Suzuki
(2004) claim that $\gamma$ is **constant over time**.

Japan is the original benchmark: $\gamma$ stability in the JMA catalog
is the primary replication test. If $\mu_\gamma^{\text{JP}} \approx 2.2$
and $\sigma_\gamma^{\text{JP}}$ is small, the replication succeeds.
Comparison with Italy and US reveals whether temporal stability is a
universal property of seismic networks or specific to Japanese tectonics.

In [ ]:
def _gamma_timeseries(df: pd.DataFrame, target_crs: str,
                      cell_size_km: float = 10.0, k_min: float = 10.0,
                      min_events: int = 500) -> pd.DataFrame:
    records = []
    for year, grp in df.groupby("year"):
        if len(grp) < min_events:
            continue
        G_yr = build_abe_suzuki_network(
            grp.sort_values("time").reset_index(drop=True),
            cell_size_km=cell_size_km, target_crs=target_crs,
        )
        G_yr_und = G_yr.to_undirected()
        G_yr_und.remove_edges_from(nx.selfloop_edges(G_yr_und))
        if G_yr_und.number_of_nodes() < 10:
            continue
        degs = [d for _, d in G_yr_und.degree(weight="weight") if d >= k_min]
        if len(degs) < 10:
            continue
        records.append({"year": int(year), "gamma": estimate_gamma_mle(degs, k_min=k_min),
                        "n_events": len(grp)})
    return pd.DataFrame(records)

print("\nComputing γ(t) for Italy...")
df_gamma_it = _gamma_timeseries(df_it, TARGET_CRS_IT)
print("Computing γ(t) for US...")
df_gamma_us = _gamma_timeseries(df_us, TARGET_CRS_US)
print("Computing γ(t) for Japan...")
df_gamma_jp = _gamma_timeseries(df_jp, TARGET_CRS_JP)

fig, ax = plt.subplots(figsize=(13, 5))
for df_g, col, marker, label in [
    (df_gamma_it, C_IT, "o", "Italy (INGV)"),
    (df_gamma_us, C_US, "s", "US (USGS)"),
    (df_gamma_jp, C_JP, "^", "Japan (JMA)"),
]:
    ax.plot(df_g["year"], df_g["gamma"], f"{marker}-", color=col, lw=2, ms=6, label=label)
    mu = df_g["gamma"].mean()
    ax.axhline(mu, color=col, ls="--", lw=1.2, alpha=0.6, label=f"{label} mean γ = {mu:.3f}")

ax.axhline(2.2, color="black", ls=":", lw=1.5, alpha=0.5, label="Abe & Suzuki (2004) γ ≈ 2.2")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("γ (MLE, 10 km network)", fontsize=12)
ax.set_title("Temporal Evolution of γ(t): Italy / US / Japan\n"
             "(Abe & Suzuki 2004: γ should be constant)", fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_gamma_timeseries_comparison")
plt.show()

df_gamma_summary = pd.DataFrame({
    "Catalog":         ["Italy (INGV)", "US (USGS)", "Japan (JMA)"],
    "γ mean":          [round(df_gamma_it["gamma"].mean(), 3),
                        round(df_gamma_us["gamma"].mean(), 3),
                        round(df_gamma_jp["gamma"].mean(), 3)],
    "γ std":           [round(df_gamma_it["gamma"].std(), 3),
                        round(df_gamma_us["gamma"].std(), 3),
                        round(df_gamma_jp["gamma"].std(), 3)],
    "γ min":           [round(df_gamma_it["gamma"].min(), 3),
                        round(df_gamma_us["gamma"].min(), 3),
                        round(df_gamma_jp["gamma"].min(), 3)],
    "γ max":           [round(df_gamma_it["gamma"].max(), 3),
                        round(df_gamma_us["gamma"].max(), 3),
                        round(df_gamma_jp["gamma"].max(), 3)],
    "Years with data": [len(df_gamma_it), len(df_gamma_us), len(df_gamma_jp)],
})
print("\n=== γ(t) Summary ===")
display(df_gamma_summary)

## Scaling Laws — kmax and Degree Moments

For a scale-free network with exponent $\gamma$, theory predicts
$k_{\max} \sim N^{1/(\gamma-1)}$; for $\gamma < 3$ the second moment
$\langle k^2 \rangle$ diverges with $N$ while $\langle k \rangle$ converges.

Adding Japan extends the dynamic range of $N$: with a much larger catalog,
Japan samples a wider range of cumulative network sizes. If all three catalogs
share the same log-log slope for $k_{\max}$ vs $N$, scaling universality holds
across three distinct seismotectonic regimes.

In [ ]:
def _cumulative_scaling(df: pd.DataFrame, target_crs: str,
                        cell_size_km: float = 10.0, k_min: float = 10.0,
                        sample_years: int = 3) -> pd.DataFrame:
    years = sorted(df["year"].unique())
    records = []
    for end_year in years[::sample_years]:
        subset = df[df["year"] <= end_year].sort_values("time").reset_index(drop=True)
        G = build_abe_suzuki_network(subset, cell_size_km=cell_size_km, target_crs=target_crs)
        G_und = G.to_undirected()
        G_und.remove_edges_from(nx.selfloop_edges(G_und))
        degs = np.array([d for _, d in G_und.degree(weight="weight") if d > 0], dtype=float)
        if len(degs) < 10:
            continue
        tail = degs[degs >= k_min]
        gamma = estimate_gamma_mle(tail.tolist(), k_min=k_min) if len(tail) > 10 else float("nan")
        records.append({"year": int(end_year), "N": G_und.number_of_nodes(),
                        "kmax": float(degs.max()), "k_mean": float(degs.mean()),
                        "k2_mean": float((degs**2).mean()), "gamma": gamma})
    return pd.DataFrame(records)

print("\nComputing cumulative scaling for Italy...")
df_scale_it = _cumulative_scaling(df_it, TARGET_CRS_IT)
print("Computing cumulative scaling for US...")
df_scale_us = _cumulative_scaling(df_us, TARGET_CRS_US)
print("Computing cumulative scaling for Japan...")
df_scale_jp = _cumulative_scaling(df_jp, TARGET_CRS_JP)

fig, ax = plt.subplots(figsize=(8, 5))
for label, df_s, col, marker in [
    ("Italy",  df_scale_it, C_IT, "o"),
    ("US",     df_scale_us, C_US, "s"),
    ("Japan",  df_scale_jp, C_JP, "^"),
]:
    ax.scatter(df_s["N"], df_s["kmax"], color=col, s=60, marker=marker,
               label=label, zorder=3)
    valid = df_s["N"] > 0
    if valid.sum() > 2:
        slope, intercept, *_ = linregress(np.log10(df_s.loc[valid, "N"]),
                                          np.log10(df_s.loc[valid, "kmax"]))
        x_fit = np.linspace(df_s["N"].min(), df_s["N"].max(), 100)
        ax.plot(x_fit, 10**intercept * x_fit**slope, "--", color=col, lw=1.8,
                label=f"{label} slope={slope:.2f}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("N (nodes)", fontsize=12)
ax.set_ylabel("kmax", fontsize=12)
ax.set_title(r"Largest Degree Scaling: $k_{max} \sim N^{1/(\gamma-1)}$", fontsize=13)
ax.legend(fontsize=10, ncol=2)
ax.grid(True, which="both", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
savefig("3cat_scaling_laws_kmax")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, df_s, col, marker in [
    ("Italy",  df_scale_it, C_IT, "o-"),
    ("US",     df_scale_us, C_US, "s-"),
    ("Japan",  df_scale_jp, C_JP, "^-"),
]:
    axes[0].plot(df_s["year"], df_s["k_mean"],  marker, color=col, lw=2, ms=5, label=label)
    axes[1].plot(df_s["year"], df_s["k2_mean"], marker, color=col, lw=2, ms=5, label=label)
axes[0].set_xlabel("Year", fontsize=11); axes[0].set_ylabel("⟨k⟩", fontsize=12)
axes[0].set_title("Mean Degree ⟨k⟩ — should stabilise", fontsize=12)
axes[0].legend(fontsize=10); axes[0].grid(ls="--", alpha=0.3)
axes[1].set_xlabel("Year", fontsize=11); axes[1].set_ylabel("⟨k²⟩", fontsize=12)
axes[1].set_title("Second Moment ⟨k²⟩ — diverges if γ < 3", fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(ls="--", alpha=0.3)
fig.suptitle("Scale-Free Moment Analysis: Italy / US / Japan", fontsize=13)
plt.tight_layout()
savefig("3cat_scaling_laws_moments")
plt.show()

## Link Prediction as Seismic Forecasting

Train on 1985–2022, test on new cell-pair connections appearing in 2023–2025.
Predictors: CN, AA, RA, Jaccard, Katz, PPR.

Japan is the seismically richest catalog — the 2023–2025 test window includes
aftershocks of the 2024 Noto Peninsula earthquake (M7.5, 1 Jan 2024).
Higher test-window activity should translate to more test edges and potentially
higher AUC scores if the network topology captures fault-zone clustering.

In [ ]:
TRAIN_END = 2022
print(f"\nBuilding train networks (up to {TRAIN_END}) for link prediction...")

G_it_train = build_abe_suzuki_network(
    df_it[df_it["year"] <= TRAIN_END].sort_values("time").reset_index(drop=True),
    cell_size_km=10, target_crs=TARGET_CRS_IT)
G_us_train = build_abe_suzuki_network(
    df_us[df_us["year"] <= TRAIN_END].sort_values("time").reset_index(drop=True),
    cell_size_km=10, target_crs=TARGET_CRS_US)
G_jp_train = build_abe_suzuki_network(
    df_jp[df_jp["year"] <= TRAIN_END].sort_values("time").reset_index(drop=True),
    cell_size_km=10, target_crs=TARGET_CRS_JP)

def _to_und_simple(G: nx.DiGraph) -> nx.Graph:
    G_u = G.to_undirected()
    G_u.remove_edges_from(nx.selfloop_edges(G_u))
    return G_u

G_it_train_u = _to_und_simple(G_it_train)
G_us_train_u = _to_und_simple(G_us_train)
G_jp_train_u = _to_und_simple(G_jp_train)
G_it_full_u  = _to_und_simple(G_it_10)
G_us_full_u  = _to_und_simple(G_us_10)
G_jp_full_u  = _to_und_simple(G_jp_10)

print("Splitting edges...")
rng_lp = np.random.default_rng(SEED)
_, test_pos_it, test_neg_it = split_edges_temporal(G_it_train_u, G_it_full_u,
                                                     n_negatives=LP_SAMPLE, seed=SEED)
_, test_pos_us, test_neg_us = split_edges_temporal(G_us_train_u, G_us_full_u,
                                                     n_negatives=LP_SAMPLE, seed=SEED)
_, test_pos_jp, test_neg_jp = split_edges_temporal(G_jp_train_u, G_jp_full_u,
                                                     n_negatives=LP_SAMPLE, seed=SEED)
if LP_SAMPLE is not None:
    for pos_list in [test_pos_it, test_pos_us, test_pos_jp]:
        if len(pos_list) > LP_SAMPLE:
            idx = rng_lp.choice(len(pos_list), LP_SAMPLE, replace=False)
            pos_list[:] = [pos_list[i] for i in idx]

LP_METHODS = ["CN", "AA", "RA", "Jaccard", "Katz", "PPR"]

print("Evaluating link prediction (Italy)...")
df_lp_it = evaluate_predictors(G_it_train_u, test_pos_it, test_neg_it, methods=LP_METHODS)
print("Evaluating link prediction (US)...")
df_lp_us = evaluate_predictors(G_us_train_u, test_pos_us, test_neg_us, methods=LP_METHODS)
print("Evaluating link prediction (Japan)...")
df_lp_jp = evaluate_predictors(G_jp_train_u, test_pos_jp, test_neg_jp, methods=LP_METHODS)

print(f"\n=== Link Prediction AUC — Italy (train ≤ {TRAIN_END}) ==="); display(df_lp_it)
print(f"\n=== Link Prediction AUC — US (train ≤ {TRAIN_END}) ===");    display(df_lp_us)
print(f"\n=== Link Prediction AUC — Japan (train ≤ {TRAIN_END}) ==="); display(df_lp_jp)

plot_auc_comparison(df_lp_it, title=f"Italy (train ≤ {TRAIN_END})")
plot_auc_comparison(df_lp_us, title=f"US (train ≤ {TRAIN_END})")
plot_auc_comparison(df_lp_jp, title=f"Japan (train ≤ {TRAIN_END})")

df_lp_cmp = (df_lp_it[["method", "AUC"]].rename(columns={"AUC": "Italy AUC"})
             .merge(df_lp_us[["method", "AUC"]].rename(columns={"AUC": "US AUC"}), on="method")
             .merge(df_lp_jp[["method", "AUC"]].rename(columns={"AUC": "Japan AUC"}), on="method"))
print("\n=== Link Prediction AUC Comparison ===")
display(df_lp_cmp)

## Export Comparison Summary

In [ ]:
out_path = RESULTS_DIR / "data" / "3cat_comparison_summary.csv"
df_macro.reset_index().to_csv(out_path, index=False)
print(f"\n3-catalog summary saved → {out_path}")